In [ ]:
"""
<프롬프트 템플릿(Prompt Templates) 개념>
 * 사용자의 입력과 매개변수를 언어 모델에 대한 "지침(챗봇 역할과 사용자 질문)으로 변환하는 역할"을 수행
   - 이를 통해, 모델이 상황을 효과적으로 파악하여,
   - 관련성 높은 일관된 텍스트 출력을 생성할 수 있도록 "유도할 수 있음"


 * 입력은 딕셔너리{} 형태(타입)으로 정의함
   - 딕셔너리의 키(key)는 프롬프트 내의 변수로 사용되며,
   - 이 변수는 실제 값으로 채워지게됨


 * 사용 라이브러리 : from langchain_core.prompts




<프롬프트 템플릿 유형>
 1. 문자열 프롬프트 템플릿(string Prompt Templates)
   - 단일 문자열 형태의 프롬프트를 생성하는데 사용됨
   - 간단한 입력에 사용됨
   - 사용 클래스 및 함수
     : PromptTemplate.from_template()




 2. 챗 프롬프트 템플릿(Chat Prompt Templates)
   - 대화형 AI 모델과 상호작용(요청과 응답) 하는데 필요한,
     -> 메시지 시퀀스를 생성하는 구조임.




   - 메시지 시퀀스 주요 구성요소
     -> 시스템 메시지(system)
        : AI의 역할을 정의하는 영역
        : AI가 어떤 종류의 "응답"을 제공해야 하는지를 명확히 정의




     -> 사용자 메시지(user)
        : 사용자가 AI에게 "요청"하는 영역
          AI에게 특정 정보를 "요청"함




   - 사용 클래스 및 함수
     : ChatPromptTemplate.from_messages([
          ("system", "응답 역할 정의"),
          ("user", "요청 내용(질문) 정의")
       ])




 3. 메시지 자리 표시자(Messages Place Holder)
   - 템플릿 내에서 동적(자동)으로 메시지를 삽입하는 역할을 수행함
   - 여러 개의 대화 메시지(채팅 기록)가 통째로 들어갈 빈칸"을 만들어두는 전용 블록임
   - 시스템 메시지(system)와 함께 사용함
   
   - 사용 클래스 및 함수
     -> 아래 코드에서 "message"가 자리 표시자가 됨




     (사용방법-1)
     : ChatPromptTemplate.from_messages([
          ("system", "응답 역할 정의"),
          MessagesPlaceholder("message")
       ])




     (사용방법-2)
     : ChatPromptTemplate.from_messages([
          ("system", "응답 역할 정의"),
          ("placeholder", "{message}")
       ])
"""


In [3]:
### 1. 문자열 프롬프트 템플릿
from langchain_core.prompts import PromptTemplate

### 2. 챗 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate

### 3. 메시지 자리 표시자
from langchain_core.prompts import MessagesPlaceholder

### 4. 메시지 자리 표시자에 전달할 실제 메시지 처리 라이브러리
#   - 사람이 입력한 행위를 대신하는 라이브러리
from langchain_core.messages import HumanMessage

# 올라마 라이브러리
from langchain_ollama import ChatOllama

### 응답 출력 파서 라이브러리
from langchain_core.output_parsers import StrOutputParser

In [ ]:
"""
<LLM 모델 처리 순서>
1. 템플릿 정의
2. 모델 정의(mistral)
3. 출력 정의
4. 랭체인(파이프라인 연결) 정의
5. 랭체인 실행(시작점)
6. 결과 출력
"""

### 1. 문자열 프롬프트 템플릿(from_template) 사용

In [23]:
"""
* from_template() 사용
  - 역할 : "주제 {topic}에 대한 개념을 짧게 요약 해줘."
  - 역할에 맞게 from_template()을 정의한 후, 최종 질문과 답변을 출력하기
  - 출력 형태
    [사용자] : 질문 출력
    [챗봇 답변] : 답변 출력
"""
################### [랭체인(LangChain) 구조 정의(1~5번까지)] ####################
# 1. 프롬프트 정의
role_str = "주제 {topic}에 대한 개념을 짧게 요약 해줘."

# 2. 프롬프트 템플릿 정의
prompt_template = ChatPromptTemplate.from_template(role_str)

# 3. 모델 정의

llm = ChatOllama(
    model="mistral",
    # - temperature : 값이 낮아질수록 환각 줄어듦 
    temperature=0.1,
    max_tokens=100
)

# 4. 출력 정의
parser = StrOutputParser()

# 5. 랭체인 파이프라인 연결 정의
chain = prompt_template | llm | parser

###########################[랭체인(LangChain) 실행 (6번)]#########################

# 6. 랭체인 실행(시작점)
# - 사용자로부터 입력 받는 값을 저장할 변수 정의(어떤 질문이 들어올지 모름)
#  -> 실제 서비스시에는 사용자 키보드로 입력된 값을 받아서 처리해야함
#  -> message = input(질문을 입력해주세요)
question = input("질문을 입력해주세요")

# - 템플릿에 전달할 딕셔너리 정의
#  -> topic key 변수는 템플릿에서 실제로 사용하는 변수명과 동일해야함
user_message = {"topic" : question}

# - 실제로 체인 실행 시작 시키기
#  -> chain의 가장 첫번째 기능(prompt)에게 사용자의 질문을 전달하게됨
result = chain.invoke(user_message)

###########################[랭체인(LangChain) 출력 (7번)]#########################
# 7. 출력
print(f"[사용자] : {question}")
print(f"[챗봇 답변] : {result}")

[사용자] : 인공지능이 뭐야 설명해줘
[챗봇 답변] :  인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간과 같은 지능적인 행동을 수행하는 기술입니다. 이는 학습, 이해, 추론, 의사 결정 등 인간의 지능적 능력을 모방하거나 혹은 새로운 방법으로 구현한 것을 말합니다. AI는 다양한 분야에서 활용되며, 예를 들어 자율주행 차량, 음성 인식, 이미지 인식, 로봇 제어 등에 사용됩니다.


In [46]:
"""
사용자로부터 계속 질문을 받고, 챗봇이 응답할 수 있도록 프로그래밍 해주세요.
- 출력은 질문과 답변이 계속 출력되도록 해주세요.
- 더 이상 질문을 하지 않고 종료하고자 할 때는 "quit"를 질문으로 입력 받으면 
    "다음에 또 만나요~"를 출력하고 종료 시키기
"""
# 1. 템플릿 정의하기
role_str = "{topic}에 대해서 짧게 한줄로 답변 해줘."
prompt_template = ChatPromptTemplate.from_template(role_str)
prompt_template

# 2. 모델 정의하기
llm = ChatOllama(
    model="mistral",
    temperature=0.1,
    max_tokens=100
)
llm

# 3. 출력 정의하기
parser = StrOutputParser()
parser

# 4. 랭체인 연결하기
chain = prompt_template | llm | parser
chain

# 계속 돌아가도록 무한 반복 
while True:
   
    # 5. 사용자에게 받을 질문 정의하기
    message = input("질문을 입력하세요(종료하려면 quit를 입력해 주세요!!) : ")
    if message == "quit":
        print("다음에 또 만나요 ~~~!!!")
        break

    if message == "":
        print("질문을 작성해 주셔야 합니다")
        # continue : 더 이상 아래 프로그램을 실행하지 말고 다음 반복으로 넘아가서 실행
        continue

    # 6. 출력할 딕셔너리 정의
    user_message = {"topic" : message}

    # 7. 생성하기
    answer = chain.invoke(user_message)

    # 8. 출력하기
    print(f"[질문] : {message}")
    print(f"[답변] : {answer}")
    

질문을 작성해 주셔야 합니다
질문을 작성해 주셔야 합니다
질문을 작성해 주셔야 합니다
질문을 작성해 주셔야 합니다
[질문] : 하이
[답변] :  Hi는 영어에서 인사하는 말로, 대화의 시작을 의미합니다. (Hello는 영어에서 인사하는 말로, 대화의 시작을 의미합니다.)
다음에 또 만나요 ~~~!!!


### 2. 챗 프롬프트 테플릿 사용

In [21]:
"""
<챗 프롬프트 템플릿(Chat Prompt Templates)>
  - 대화형 AI 모델과 상호작용(요청과 응답)을 수행하는데 필요한,
    -> 메시지 시퀀스를 생성하는 구조로 처리됨
  - 메시지 시퀀스의 의미 : 사전 system에게 역할 또는 규칙을 정의한 내용을 기준으로
                      : 사용자의 요청을 처리하게 됨
                      : 모든 메시지(요청/응답 과정의 데이터)를 시퀀스로 순서에 맞게 구조화하여 처리함


  - 사용 클래스 및 함수
    ChatPromptTemplate.from_messages([("system", "역할 또는 규칙 등 정의"),
                                      ("user", "사용자 요청 내용 정의")])
"""


' 인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간과 같은 인식, 학습, 의사 결정 능력을 가지도록 설계된 소프트웨어이다. AI는 자연어 처리, 이미지 인식, 로봇 제어, 자율 주행 등 다양한 분야에서 활용되며, 기계 학습, 딥 러닝,  rule-based system 등 다양한 기술을 활용하여 개발된다. AI는 인간의 지능을 모방하거나 인간과 같은 능력을 발전시키는 데 중점을 둔 연구 분야이며, 앞으로 발전하면 사회, 경제, 생활 등 다양한 영역에 영향을 미칠 것으로 예상된다.'

In [ ]:
"""
1. 챗봇(system) 역할 부여 : "요약 전문가" 역할 부여시키기
2. 사용자(user) 질문 정의 : "주제 {topic}에 대해 짧게 요약해줘
- 1번과 2번을 ChatPromptTemplate.from_messages() 함수를 이용해서 챗 프롬프트 템플릿 정의
- 모델을 통해 응답 결과 출력하기

** 출력 형태 : 질문과 답변을 모두 출력해 주세요..
- 단, 계속 질문을 받을 수 있도록 처리해 주세요.. 
"""

In [ ]:
# 프롬프트 정의
sys_cmd = "너는 요약 전문가야. 나에게 설명해줘야 할 대상을 잘 요약해서 설명해주면 돼."

prompt = "주제 {topic}에 대해 짧게 요약해줘"
prompt_template = ChatPromptTemplate.from_messages([
    ("system", sys_cmd),
    ("user", prompt)
])

# 랭체인 정의
chain = prompt_template | llm | parser

while True:
    message = input("질문을 입력해주세요")
    if message == "quit":
        break

    if message == "":
        print("질문을 입력하세요!")
        continue

    user_message = {"topic":message}

    result = chain.invoke(user_message)

    print(f"[질문] : {message}")
    print(f"[답변] : {result}")

[질문] : 하이
[답변] :  주제 하이(Subject High)는 한국어 학습을 위한 온라인 플랫폼으로, 학습자가 자신의 능력 수준에 맞는 강의와 문제를 선택하여 학습할 수 있는 서비스입니다. 주제 하이는 다양한 학습 방법을 제공하며, 기초 학습부터 고급 학습, 시험 준비 학습, 영어 학습 등 다양한 분야에서 학습을 할 수 있습니다. 또한, 학습 과정에서 자동 피드백을 제공하여 학습자의 성과를 향상시키고, 학습 과정을 추적하고 진행 상황을 파악할 수 있습니다. 주제 하이는 학습자의 개인화된 학습 경험을 제공하여 효과적인 한국어 학습을 가능하게 합니다.


### 3. 메시지 자리 표시자 사용

In [ ]:
"""
 - 템플릿 내에서 동적으로 메시지를 삽입하는 역할 수행
 - 시스템(system) 메시지와 함께 사용됨


 <사용방법>
  1. ChatPromptTemplate.from_messages([
        ("system", "역할 또는 규칙 정의"),


        ### MessagesPlaceholder() : 메시지 자리 표시자-> message
        MessagesPlaceholder("message")
     ])
     invoke({"message" : [HumanMessage(content="사용자 요청 메시지")]})
 
  2. ChatPromptTemplate.from_messages([
        ("system", "역할 또는 규칙 정의),


        ### placeholder : 메시지 자리 표시자-> message
        ("placeholder", "{message}")
     ])
     invoke({"message" : [HumanMessage(content="사용자 요청 메시지")]})
"""


In [56]:
### 1. 프롬프트 정의
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "지금부터 너는 요약 전문가야."),

    ### MessagesPlaceholder() : 메시지 자리 표시자-> message
    MessagesPlaceholder("message")
])

### 2. 모델 정의
llm = ChatOllama(model="mistral", temperature=0.1, max_tokens=100)

### 3. 출력 파서 정의
parser = StrOutputParser()

### 4. 체인 연결
chain = prompt_template | llm | parser
chain

### 5. 체인 실행하기
# - 사용자로부터 질문(message) 입력 받기
message = "생성형 모델의 개념에 대해서 한글로 요약해줘."

# - 모델에 요청하기 위해 딕셔너리 타입으로 정의
# -> HumanMesssage : MessagesPlaceholder 메시지 자리표에 들어갈 데이터 정의(질문이 들어감)
user_message = {"message" : [HumanMessage(content=message)]}

# - 체인실행
result = chain.invoke(user_message)

print(f"[질문] : {message}")
print(f"[답변] : {result}")

[질문] : 생성형 모델의 개념에 대해서 한글로 요약해줘.
[답변] :  생성형 모델(Generative Model)은 데이터 분석 및 학습 중에 데이터 생성 또는 예측 데이터 생성을 목적으로 하는 모델입니다. 이 모델은 주어진 데이터를 학습하여 새로운 데이터를 생성하거나, 주어진 데이터의 특성을 이해하여 새로운 데이터를 예측하는 데 사용됩니다. 예를 들어, 이미지 생성, 음성 합성, 문장 생성 등에 사용됩니다. 생성형 모델은 학습 데이터의 분포를 학습하여 새로운 데이터를 생성하거나 예측하는 데 사용되며, 이를 통해 데이터 생성, 데이터 증강, 데이터 분석 등의 다양한 분야에서 활용됩니다.


In [57]:
rs = prompt_template.invoke(user_message)
rs

ChatPromptValue(messages=[SystemMessage(content='지금부터 너는 요약 전문가야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='생성형 모델의 개념에 대해서 한글로 요약해줘.', additional_kwargs={}, response_metadata={})])